# Custom Model Run example with MLP and no trace logs

In [ ]:
from __future__ import annotations

import copy
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from pydart import (
    ArithmeticIntensityMetric,
    Evaluator,
    ModelSpec,
    Node,
    Profiler,
    Task,
    Taskset,
)

# from pydart.paths import CUSTOM_OUTPUTS_DIR

# CUSTOM_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

class CustomMLP(nn.Module):
    def __init__(
        self,
        in_features: int = 128,
        hidden: int = 256,
        out_features: int = 64,
    ) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_features),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def build_custom_dataloader() -> DataLoader:
    inputs = torch.randn(100, 128)
    targets = torch.randint(0, 64, (100,))
    dataset = TensorDataset(inputs, targets)
    return DataLoader(dataset, batch_size=16)


def build_custom_spec() -> ModelSpec:
    return ModelSpec(
        name="custom_mlp",
        builder=lambda: CustomMLP(),
        input_size=(128,),
        batch_size=16,
        num_samples=100,
        num_classes=64,
        dataloader_fn=build_custom_dataloader,
    )


def create_task_from_spec(
    task_id: str,
    spec: ModelSpec,
    profiler: Profiler,
    metric: ArithmeticIntensityMetric,
) -> Task:
    model = spec.build_model().eval()
    input_data = spec.build_input_batch()

    return Task(
        task_id=task_id,
        model=model,
        input_data=input_data,
        model_name=spec.name,
        profiler=profiler,
        load_metric=metric,
    )


def main() -> None:
    # 1. Discover workers
    nodes = Node.discover_shared_gpu_workers(num_workers=2, gpu_id=0)
    print(f"Using {len(nodes)} shared-GPU workers: {nodes}")

    # 2. Create profiler and metric
    profiler = Profiler(mode="init")
    metric = ArithmeticIntensityMetric()

    # 3. Build custom spec
    custom_spec = build_custom_spec()

    # 4. Create tasks
    tasks = []
    for i in range(20):
        tasks.append(
            create_task_from_spec(
                task_id=f"custom_task_{i+1}",
                spec=custom_spec,
                profiler=profiler,
                metric=metric,
            )
        )

    # 5. Profile every task on every node
    for task in tasks:
        for node in nodes:
            inp_copy = copy.deepcopy(task.input_data)
            profiler.profile_model(
                model=task.model,
                input_data=inp_copy,
                node=node,
                task_id=task.task_id,
            )
            time.sleep(0.05)

    # 6. Populate profile records
    for task in tasks:
        task.populate_profile_records()

    # 7. Build taskset and evaluator
    taskset = Taskset(tasks, nodes)
    evaluator = Evaluator(taskset, profiler)

    # 8. Run baselines / PyDart execution
    evaluator.run_naive_execution()
    evaluator.run_parallel_execution()

    # 9. Compare outputs and report performance
    evaluator.compare_outputs()
    evaluator.analyze_speedup_throughput()

    # 10. Stop workers
    for node in nodes:
        node.stop()


# if __name__ == "__main__":
#     main()

Using 2 shared-GPU workers: [Node(W0, cpus=(0,), gpu=0), Node(W1, cpus=(1,), gpu=0)]
[Profiler] Starting profiling for Task 'custom_task_1' on W0 (device=cpu).
[Profiler] Profiling complete. Data saved to profiling_results.csv.
[Profiler] Starting profiling for Task 'custom_task_1' on W1 (device=cpu).
[Profiler] Profiling complete. Data saved to profiling_results.csv.
[Profiler] Reused cached profiling data for CustomMLP on W0.
[Profiler] Reused cached profiling data for CustomMLP on W1.
[Profiler] Reused cached profiling data for CustomMLP on W0.
[Profiler] Reused cached profiling data for CustomMLP on W1.
[Profiler] Reused cached profiling data for CustomMLP on W0.
[Profiler] Reused cached profiling data for CustomMLP on W1.
[Profiler] Reused cached profiling data for CustomMLP on W0.
[Profiler] Reused cached profiling data for CustomMLP on W1.
[Profiler] Reused cached profiling data for CustomMLP on W0.
[Profiler] Reused cached profiling data for CustomMLP on W1.
[Profiler] Reused c

# Custom Model Run example with MLP and with trace logs

In [ ]:
from __future__ import annotations

import copy
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from viztracer import VizTracer

from pydart import (
    ArithmeticIntensityMetric,
    Evaluator,
    ModelSpec,
    Node,
    Profiler,
    Task,
    Taskset,
)

from pydart.paths import CUSTOM_OUTPUTS_DIR

CUSTOM_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)


class CustomMLP(nn.Module):
    def __init__(
        self,
        in_features: int = 128,
        hidden: int = 256,
        out_features: int = 64,
    ) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_features),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def build_custom_dataloader() -> DataLoader:
    inputs = torch.randn(100, 128)
    targets = torch.randint(0, 64, (100,))
    dataset = TensorDataset(inputs, targets)
    return DataLoader(dataset, batch_size=16)


def build_custom_spec() -> ModelSpec:
    return ModelSpec(
        name="custom_mlp",
        builder=lambda: CustomMLP(),
        input_size=(128,),
        batch_size=16,
        num_samples=100,
        num_classes=64,
        dataloader_fn=build_custom_dataloader,
    )


def create_task_from_spec(
    task_id: str,
    spec: ModelSpec,
    profiler: Profiler,
    metric: ArithmeticIntensityMetric,
) -> Task:
    model = spec.build_model().eval()
    input_data = spec.build_input_batch()

    return Task(
        task_id=task_id,
        model=model,
        input_data=input_data,
        model_name=model.__class__.__name__,
        profiler=profiler,
        load_metric=metric,
    )


def main() -> None:
    nodes = Node.discover_shared_gpu_workers(num_workers=2, gpu_id=0)
    print(f"Using {len(nodes)} shared-GPU workers: {nodes}")

    profiler = Profiler(mode="init")
    metric = ArithmeticIntensityMetric()

    custom_spec = build_custom_spec()

    tasks = []
    for i in range(5):
        tasks.append(
            create_task_from_spec(
                task_id=f"custom_task_{i+1}",
                spec=custom_spec,
                profiler=profiler,
                metric=metric,
            )
        )

    for task in tasks:
        for node in nodes:
            inp_copy = copy.deepcopy(task.input_data)
            profiler.profile_model(
                model=task.model,
                input_data=inp_copy,
                node=node,
                task_id=task.task_id,
            )
            time.sleep(0.05)

    for task in tasks:
        task.populate_profile_records()

    taskset = Taskset(tasks, nodes)
    evaluator = Evaluator(taskset, profiler)

    naive_trace = CUSTOM_OUTPUTS_DIR / "custom_mlp_naive_trace.html"
    parallel_trace = CUSTOM_OUTPUTS_DIR / "custom_mlp_parallel_trace.html"

    tracer_naive = VizTracer(output_file=str(naive_trace))
    tracer_naive.start()
    evaluator.run_naive_execution()
    tracer_naive.stop()
    tracer_naive.save()

    tracer_parallel = VizTracer(output_file=str(parallel_trace))
    tracer_parallel.start()
    evaluator.run_parallel_execution()
    tracer_parallel.stop()
    tracer_parallel.save()

    evaluator.compare_outputs()
    evaluator.analyze_speedup_throughput()

    print(f"Naive trace saved to: {naive_trace}")
    print(f"Parallel trace saved to: {parallel_trace}")
    print(f"All custom outputs stored in: {CUSTOM_OUTPUTS_DIR.resolve()}")

    for node in nodes:
        node.stop()


if __name__ == "__main__":
    main()

Using 2 shared-GPU workers: [Node(W0, cpus=(0,), gpu=0), Node(W1, cpus=(1,), gpu=0)]
[Profiler] Starting profiling for Task 'custom_task_1' on W0 (device=cpu).
[Profiler] Profiling complete. Data saved to profiling_results.csv.
[Profiler] Starting profiling for Task 'custom_task_1' on W1 (device=cpu).
[Profiler] Profiling complete. Data saved to profiling_results.csv.
[Profiler] Reused cached profiling data for CustomMLP on W0.
[Profiler] Reused cached profiling data for CustomMLP on W1.
[Profiler] Reused cached profiling data for CustomMLP on W0.
[Profiler] Reused cached profiling data for CustomMLP on W1.
[Profiler] Reused cached profiling data for CustomMLP on W0.
[Profiler] Reused cached profiling data for CustomMLP on W1.
[Profiler] Reused cached profiling data for CustomMLP on W0.
[Profiler] Reused cached profiling data for CustomMLP on W1.
[Schedule] custom_task_1
  custom_task_1-stage-1: -> W0 (pred run nan µs)
  custom_task_1-stage-2: -> W0 (pred run 1575.00 µs)
[Schedule] cus

# Custom Models with Simple MLP and Swin Transformer and trace logs

In [2]:
from __future__ import annotations

import copy
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models
from viztracer import VizTracer


from pydart import (
    ArithmeticIntensityMetric,
    Evaluator,
    ModelSpec,
    Node,
    Profiler,
    Task,
    Taskset,
)
from pydart.utils import set_seed
from pydart.paths import CUSTOM_OUTPUTS_DIR


# REPO_ROOT = Path.cwd().parents[0]
# OUTPUTS_DIR = REPO_ROOT / "outputs_custom"
CUSTOM_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)


class CustomMLP(nn.Module):
    def __init__(
        self,
        in_features: int = 128,
        hidden: int = 2048,
        out_features: int = 64,
    ) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_features),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DemoSwinTiny(nn.Module):
    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.backbone = models.swin_t(weights=None)
        in_features = self.backbone.head.in_features
        self.backbone.head = nn.Linear(in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


def build_mlp_dataloader() -> DataLoader:
    inputs = torch.randn(100, 128)
    targets = torch.randint(0, 64, (100,))
    dataset = TensorDataset(inputs, targets)
    return DataLoader(dataset, batch_size=16)


def build_swin_dataloader() -> DataLoader:
    inputs = torch.randn(32, 3, 224, 224)
    targets = torch.randint(0, 10, (32,))
    dataset = TensorDataset(inputs, targets)
    return DataLoader(dataset, batch_size=2)


def build_custom_mlp_spec() -> ModelSpec:
    return ModelSpec(
        name="custom_mlp",
        builder=lambda: CustomMLP(),
        input_size=(128,),
        batch_size=16,
        num_samples=100,
        num_classes=64,
        dataloader_fn=build_mlp_dataloader,
    )


def build_swin_spec() -> ModelSpec:
    return ModelSpec(
        name="swin_tiny",
        builder=lambda: DemoSwinTiny(num_classes=10),
        input_size=(3, 224, 224),
        batch_size=2,
        num_samples=32,
        num_classes=10,
        dataloader_fn=build_swin_dataloader,
    )


def create_task_from_spec(
    task_id: str,
    spec: ModelSpec,
    profiler: Profiler,
    metric: ArithmeticIntensityMetric,
) -> Task:
    model = spec.build_model().eval()
    input_data = spec.build_input_batch()

    return Task(
        task_id=task_id,
        model=model,
        input_data=input_data,
        model_name=model.__class__.__name__,
        profiler=profiler,
        load_metric=metric,
    )


def main() -> None:
    set_seed(42)

    nodes = Node.discover_shared_gpu_workers(num_workers=2, gpu_id=0)
    print(f"Using {len(nodes)} shared-GPU workers: {nodes}")

    profiler = Profiler(mode="init")
    metric = ArithmeticIntensityMetric()

    mlp_spec = build_custom_mlp_spec()
    swin_spec = build_swin_spec()

    tasks = []

    for i in range(2):
        tasks.append(
            create_task_from_spec(
                task_id=f"mlp_task_{i+1}",
                spec=mlp_spec,
                profiler=profiler,
                metric=metric,
            )
        )

    for i in range(2):
        tasks.append(
            create_task_from_spec(
                task_id=f"swin_task_{i+1}",
                spec=swin_spec,
                profiler=profiler,
                metric=metric,
            )
        )

    for task in tasks:
        for node in nodes:
            inp_copy = copy.deepcopy(task.input_data)
            profiler.profile_model(
                model=task.model,
                input_data=inp_copy,
                node=node,
                task_id=task.task_id,
            )
            time.sleep(0.05)

    for task in tasks:
        task.populate_profile_records()

    taskset = Taskset(tasks, nodes)
    evaluator = Evaluator(taskset, profiler)

    naive_trace = CUSTOM_OUTPUTS_DIR / "custom_mix_naive_trace.html"
    parallel_trace = CUSTOM_OUTPUTS_DIR / "custom_mix_parallel_trace.html"

    tracer_naive = VizTracer(output_file=str(naive_trace))
    tracer_naive.start()
    evaluator.run_naive_execution()
    tracer_naive.stop()
    tracer_naive.save()

    tracer_parallel = VizTracer(output_file=str(parallel_trace))
    tracer_parallel.start()
    evaluator.run_parallel_execution()
    tracer_parallel.stop()
    tracer_parallel.save()

    evaluator.compare_outputs()
    evaluator.analyze_speedup_throughput()

    print(f"Naive trace saved to: {naive_trace}")
    print(f"Parallel trace saved to: {parallel_trace}")
    print(f"All custom outputs stored in: {CUSTOM_OUTPUTS_DIR.resolve()}")

    for node in nodes:
        node.stop()


if __name__ == "__main__":
    main()

Seed set to 42
Using 2 shared-GPU workers: [Node(W0, cpus=(0,), gpu=0), Node(W1, cpus=(1,), gpu=0)]
[Profiler] Starting profiling for Task 'mlp_task_1' on W0 (device=cpu).


C:\Users\EndUser\AppData\Roaming\Python\Python39\site-packages\torch\autograd\profiler.py:228: UserWarning: CUDA is not available, disabling CUDA profiling
  warn("CUDA is not available, disabling CUDA profiling")


[Profiler] Profiling complete. Data saved to profiling_results.csv.
[Profiler] Starting profiling for Task 'mlp_task_1' on W1 (device=cpu).


C:\Users\EndUser\AppData\Roaming\Python\Python39\site-packages\torch\autograd\profiler.py:228: UserWarning: CUDA is not available, disabling CUDA profiling
  warn("CUDA is not available, disabling CUDA profiling")


[Profiler] Profiling complete. Data saved to profiling_results.csv.
[Profiler] Reused cached profiling data for CustomMLP on W0.
[Profiler] Reused cached profiling data for CustomMLP on W1.
[Profiler] Starting profiling for Task 'swin_task_1' on W0 (device=cpu).


C:\Users\EndUser\AppData\Roaming\Python\Python39\site-packages\torch\autograd\profiler.py:228: UserWarning: CUDA is not available, disabling CUDA profiling
  warn("CUDA is not available, disabling CUDA profiling")


[Profiler] Profiling complete. Data saved to profiling_results.csv.
[Profiler] Starting profiling for Task 'swin_task_1' on W1 (device=cpu).


C:\Users\EndUser\AppData\Roaming\Python\Python39\site-packages\torch\autograd\profiler.py:228: UserWarning: CUDA is not available, disabling CUDA profiling
  warn("CUDA is not available, disabling CUDA profiling")


[Profiler] Profiling complete. Data saved to profiling_results.csv.
[Profiler] Reused cached profiling data for DemoSwinTiny on W0.
[Profiler] Reused cached profiling data for DemoSwinTiny on W1.
[Schedule] mlp_task_1
  mlp_task_1-stage-1: -> W0 (pred run nan µs)
  mlp_task_1-stage-2: -> W1 (pred run 12313.00 µs)
[Schedule] mlp_task_2
  mlp_task_2-stage-1: -> W0 (pred run nan µs)
  mlp_task_2-stage-2: -> W1 (pred run 885.00 µs)
[Schedule] swin_task_1
  swin_task_1-stage-1: -> W1 (pred run 6570.00 µs)
  swin_task_1-stage-2: -> W0 (pred run 2070611.00 µs)
[Schedule] swin_task_2
  swin_task_2-stage-1: -> W0 (pred run 13532.00 µs)
  swin_task_2-stage-2: -> W1 (pred run 1769262.00 µs)
[Evaluator] Starting Naive Execution.
Running on cpu
[Evaluator] Task mlp_task_1: Naive exec time: 0.0009s
[Evaluator] Task mlp_task_2: Naive exec time: 0.0006s
[Evaluator] Task swin_task_1: Naive exec time: 0.4693s
[Evaluator] Task swin_task_2: Naive exec time: 0.3822s
[Evaluator] Naive makespan: 0.8651s

Tot